<a href="https://colab.research.google.com/github/AbdelbasetKABOU/Qwen7B-NL-XML-SFT-8K/blob/main/qwen2_5_7B_A100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install required libraries

Install main dependencies needed for fine-tuning the LLM :

*   Transformers for model handling,
*   Accelerate for efficient training,
*   PEFT for parameter-efficient fine-tuning (LoRA),
*   BitsAndBytes for memory-efficient quantization.

In [ ]:
!pip -q install -U \
  "transformers>=4.44.0" \
  "accelerate>=0.33.0" \
  "peft>=0.12.0" \
  "bitsandbytes>=0.43.0"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 131.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 38.8 MB/s eta 0:00:00


### Verify environment and library versions

Import the required libraries and print versions to confirm  environment is correctly configured + CUDA is available.

In [ ]:
import torch, transformers, accelerate, peft
import bitsandbytes as bnb
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("bnb:", bnb.__version__)
print("cuda:", torch.cuda.is_available())

# torch: 2.9.0+cu126
# transformers: 4.57.3
# accelerate: 1.12.0
# peft: 0.18.0
# bnb: 0.49.1
# cuda: True

torch: 2.9.0+cu126
transformers: 4.57.3
accelerate: 1.12.0
peft: 0.18.0
bnb: 0.49.1
cuda: True


### Mount Google Drive

Mount Google Drive in Colab environment.

In [ ]:
from google.colab import drive
## drive.mount("/content/drive/notebooks/projectz_at_merce/twingpt/out/sft_min")
drive.mount("/content/drive/")

# Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


### Inspect dataset directory

List contents of the dataset directory in GDrive to verify expected files.

In [ ]:
!ls ""/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/out/sft_min

# train.jsonl  val.jsonl

train.jsonl  val.jsonl


### Prepare local working directories

Create local dirs in Colab to store processed datasets and model outputs.

In [ ]:
!mkdir -p /content/data/out/sft_min_8k
!mkdir -p /content/outputs
!ls -R /content/data

# /content/data:
# out
# /content/data/out:
# sft_min_8k
# /content/data/out/sft_min_8k:

/content/data:
out

/content/data/out:
sft_min_8k

/content/data/out/sft_min_8k:


Create directory structure.

In [ ]:
!mkdir -p /content/data/labels/sft_min/train
!mkdir -p /content/data/out/sft_min_8k

Copy dataset files to local Colab to enable faster access (during preprocessing/training).

In [ ]:
# Example: if your files are in MyDrive/datasets/sft_min_8k/
!cp -v /content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/out/sft_min/* /content/data/labels/sft_min/train
!ls -lh /content/data/labels/sft_min/train

# '/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/out/sft_min/train.jsonl' -> '/content/data/labels/sft_min/train/train.jsonl'
# '/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/out/sft_min/val.jsonl' -> '/content/data/labels/sft_min/train/val.jsonl'
# total 22M
# -rw------- 1 root root  18M Jan 15 08:59 train.jsonl
# -rw------- 1 root root 3.4M Jan 15 08:59 val.jsonl

'/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/out/sft_min/train.jsonl' -> '/content/data/labels/sft_min/train/train.jsonl'
'/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/out/sft_min/val.jsonl' -> '/content/data/labels/sft_min/train/val.jsonl'
total 22M
-rw------- 1 root root  18M Jan 15 08:59 train.jsonl
-rw------- 1 root root 3.4M Jan 15 08:59 val.jsonl


### Define dataset processing utilities

Define helper functions to :
- load the JSONL dataset,
- convert chat examples into tokenized training samples,
- build PyTorch-ready datasets.  

Preprocessing masks the prompt (system + user messages) so the model is trained only to generate assistant response (target XML output).

In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def tokenize_row(row):
    msgs = row["messages"]

    # Expect: system, user, assistant
    # We will train only on assistant content.
    # Build prompt tokens (system+user) and full tokens (system+user+assistant)
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    assistant_msgs = [m for m in msgs if m["role"] == "assistant"]
    if len(assistant_msgs) != 1:
        raise ValueError(f"Expected exactly 1 assistant message, got {len(assistant_msgs)}")

    # Prompt text (system+user), with generation prompt so the template inserts the assistant header.
    prompt_text = tok.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True
    )

    # Full text includes assistant content appended
    # (we create it by taking prompt_text + assistant content)
    # For Qwen templates, this is generally safe and keeps boundary consistent.
    full_text = prompt_text + assistant_msgs[0]["content"]

    prompt_ids = tok(prompt_text, add_special_tokens=False, truncation=True, max_length=MAX_LEN)["input_ids"]
    full = tok(full_text, add_special_tokens=False, truncation=True, max_length=MAX_LEN)

    input_ids = torch.tensor(full["input_ids"], dtype=torch.long)
    attention_mask = torch.tensor(full["attention_mask"], dtype=torch.long)

    # Labels: mask prompt, learn assistant continuation
    labels = input_ids.clone()
    cut = min(len(prompt_ids), labels.numel())
    labels[:cut] = -100

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

def build_pt(in_jsonl, out_pt, limit=None):
    rows = read_jsonl(in_jsonl)
    if limit:
        rows = rows[:limit]
    data = []
    bad = 0
    for i, r in enumerate(rows):
        try:
            data.append(tokenize_row(r))
        except Exception as e:
            bad += 1
            if bad <= 3:
                print("[WARN] skipping row", i, "reason:", str(e)[:200])
    torch.save(data, out_pt)
    print(f"[OK] saved {len(data)} examples to {out_pt} (skipped {bad})")

### Load tokenizer and define dataset paths

- Initialize tokenizer for the base model (Qwen2.5-7B-Instruct),
- set the maximum sequence length (8k tokens),
- define input dataset paths and output locations for the tokenized training/validation datasets.

In [ ]:
import os, json, torch
from transformers import AutoTokenizer

MODEL = "Qwen/Qwen2.5-7B-Instruct"
MAX_LEN = 8192  # since you mentioned 8k

train_jsonl = '/content/data/labels/sft_min/train/train.jsonl'
val_jsonl   = '/content/data/labels/sft_min/train/val.jsonl'

out_train_pt = "/content/data/out/sft_min_8k/train_8k_tokenized.pt"
out_val_pt   = "/content/data/out/sft_min_8k/val_8k_tokenized.pt"

tok = AutoTokenizer.from_pretrained(MODEL, use_fast=True, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

# /usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning:
# The secret `HF_TOKEN` does not exist in your Colab secrets.
# To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
# You will be able to reuse this secret in all of your notebooks.
# Please note that authentication is recommended but still optional to access public models or datasets.
# warnings.warn(
# tokenizer_config.json: 7.30k/? [00:00<00:00, 784kB/s]
# vocab.json: 2.78M/? [00:00<00:00, 85.8MB/s]
# merges.txt: 1.67M/? [00:00<00:00, 74.9MB/s]
# tokenizer.json: 7.03M/? [00:00<00:00, 151MB/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Process training JSONL dataset, tokenize each example, and save the resulting PyTorch dataset for use during fine-tuning.

In [ ]:
build_pt(train_jsonl, out_train_pt)
# [OK] saved 648 examples to /content/data/out/sft_min_8k/train_8k_tokenized.pt (skipped 0)

[OK] saved 648 examples to /content/data/out/sft_min_8k/train_8k_tokenized.pt (skipped 0)


* Process validation JSONL using the same tokenization pipeline
* save the resulting PyTorch dataset for evaluation.

In [ ]:
build_pt(val_jsonl, out_val_pt)
# [OK] saved 112 examples to /content/data/out/sft_min_8k/val_8k_tokenized.pt (skipped 0)

[OK] saved 112 examples to /content/data/out/sft_min_8k/val_8k_tokenized.pt (skipped 0)


- Load saved tokenized training dataset,
- Inspect the first example’s tensor shapes,
- and verify that labels are correctly masked
  
  _(prompt tokens set to `-100` so training focuses only on the assistant output)_.

In [ ]:
import torch
ds = torch.load("/content/data/out/sft_min_8k/train_8k_tokenized.pt")
print("n_examples:", len(ds))
ex = ds[0]
print({k: tuple(v.shape) for k,v in ex.items()})
print("labels masked tokens:", (ex["labels"] == -100).sum().item(), "/", ex["labels"].numel())

# n_examples: 648
# {'input_ids': (1653,), 'attention_mask': (1653,), 'labels': (1653,)}
# labels masked tokens: 152 / 1653


n_examples: 648
{'input_ids': (1653,), 'attention_mask': (1653,), 'labels': (1653,)}
labels masked tokens: 152 / 1653


Display GPU details (model, VRAM, driver/CUDA versions, current usage).

In [ ]:
!nvidia-smi

Thu Jan 15 09:00:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             44W /  400W |       5MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Load tokenized datasets and set output directory

- Configure paths for tokenized train/validation `.pt` files,
- Choose output directory (here on Google Drive for persistence), - Load datasets into memory, and
- Print basic dataset/sequence-length stats.

In [ ]:
import os, torch, math

TRAIN_PT = "/content/data/out/sft_min_8k/train_8k_tokenized.pt"
VAL_PT   = "/content/data/out/sft_min_8k/val_8k_tokenized.pt"
## OUT_DIR  = "/content/outputs/qwen2p5_sft_lora_colab"
OUT_DIR = "/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab"

assert os.path.exists(TRAIN_PT), f"Missing: {TRAIN_PT}"
assert os.path.exists(VAL_PT),   f"Missing: {VAL_PT}"
os.makedirs(OUT_DIR, exist_ok=True)

train_data = torch.load(TRAIN_PT, weights_only=False)
val_data   = torch.load(VAL_PT, weights_only=False)

print("Train examples:", len(train_data))
print("Val examples:", len(val_data))

# quick length stats
lens = [int(x["input_ids"].numel()) for x in train_data[:2000]]  # sample up to 2k
print("Sample lengths: min", min(lens), "p50", sorted(lens)[len(lens)//2], "max", max(lens))

# Train examples: 648
# Val examples: 112
# Sample lengths: min 1626 p50 6403 max 8192

Train examples: 648
Val examples: 112
Sample lengths: min 1626 p50 6403 max 8192


### Define a PyTorch Dataset + padding collator

- Wrap `.pt` files in a `Dataset` class and
- Define a custom data collator that dynamically pads `input_ids`, `attention_mask`, and `labels` to the longest sequence in each batch (with label padding set to `-100`).

In [ ]:
from dataclasses import dataclass
from typing import List, Dict
from torch.utils.data import Dataset

class TokenizedPTDataset(Dataset):
    def __init__(self, pt_path: str):
        self.data = torch.load(pt_path, weights_only=False)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]
        return {
            "input_ids": ex["input_ids"],
            "attention_mask": ex["attention_mask"],
            "labels": ex["labels"],
        }

@dataclass
class DataCollatorForCausalLMWithLabels:
    pad_token_id: int
    label_pad_id: int = -100

    def __call__(self, features: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        max_len = max(f["input_ids"].numel() for f in features)

        def pad_1d(x, pad_value):
            if x.numel() == max_len:
                return x
            pad = torch.full((max_len - x.numel(),), pad_value, dtype=x.dtype)
            return torch.cat([x, pad], dim=0)

        input_ids = torch.stack([pad_1d(f["input_ids"], self.pad_token_id) for f in features])
        attention_mask = torch.stack([pad_1d(f["attention_mask"], 0) for f in features])
        labels = torch.stack([pad_1d(f["labels"], self.label_pad_id) for f in features])

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


### Tokenizer and model setup

- Load tokenizer for the base model (Qwen2.5-7B-Instruct), ensuring that a padding token is defined for batching during training.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

MODEL = "Qwen/Qwen2.5-7B-Instruct"

# Tokenizer
tok = AutoTokenizer.from_pretrained(MODEL, use_fast=True, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token


Determine appropriate computation dtype _(balance performance and numerical stability)_.

In [ ]:
# Pick compute dtype
bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8  # A100/L4
compute_dtype = torch.bfloat16 if bf16_ok else torch.float16
print("Using compute dtype:", compute_dtype)

# Using compute dtype: torch.bfloat16

Using compute dtype: torch.bfloat16


### Load base model with 4-bit quantization (QLoRA)

- Configure BitsAndBytes 4-bit quantization and
- Load pretrained model in memory-efficient mode.

This enables fine-tuning large models on limited GPU memory while keeping training efficient.

In [ ]:
# 4-bit quant config (QLoRA)
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

model.config.use_cache = False

# Loading checkpoint shards: 100% 4/4 [00:16<00:00,  4.05s/it]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

### Prepare quantized model

Enable k-bit fine-tuning utilities and gradient checkpointing so gradients flow correctly through the 4-bit model while reducing GPU memory usage.

In [ ]:
# IMPORTANT: prepare model so k-bit + checkpointing works and grads flow
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

### Attach LoRA adapters (parameter-efficient fine-tuning)

Define + apply LoRA configuration targeting the main attention projection layers : q/k/v/o (i.e., trains only small set of adapter weights while keeping base model frozen).

In [ ]:
# LoRA config
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_cfg)

### Final training setup and parameter overview

- Ensure inputs require gradients (useful for checkpointing)
- Print how many parameters are trainable (confirm that only LoRA weights will be updated).

In [ ]:
# This helps with some checkpointing edge cases
model.enable_input_require_grads()
model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


### Configure training datasets / hyperparameters

- Instantiate train/validation `Dataset` objects + padding collator,
- Set key training hyperparameters (batch size, gradient accumulation, learning rate, max steps, and eval/save frequency).  

The effective batch size is `BATCH × GRAD_ACCUM`, and we intentionally cap `MAX_STEPS` to limit overfitting observed in earlier runs.

In [ ]:
import math
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

train_ds = TokenizedPTDataset(TRAIN_PT)
val_ds   = TokenizedPTDataset(VAL_PT)
collator = DataCollatorForCausalLMWithLabels(pad_token_id=tok.pad_token_id)

BATCH = 1
GRAD_ACCUM = 16
LR = 2e-4

# We already observed overfitting by ~200,
# no need to go to 1200 again
## MAX_STEPS = 1200 bzff
## EVAL_STEPS = 50 bzf
## SAVE_STEPS = 50
MAX_STEPS = 300
EVAL_STEPS = 25
SAVE_STEPS = 25

steps_per_epoch = math.ceil(len(train_ds) / (BATCH * GRAD_ACCUM))
print("Approx steps/epoch:", steps_per_epoch)

# Approx steps/epoch: 41

Approx steps/epoch: 41


### Launch fine-tuning with Hugging Face Trainer (QLoRA + early stopping)

- Configure `TrainingArguments` for QLoRA training (8-bit paged AdamW optimizer, gradient checkpointing, mixed precision bf16/fp16, step-based eval/save).  
- Train using `Trainer` with early stopping and automatically reload best checkpoint based on validation loss (`eval_loss`) to mitigate overfitting.

In [ ]:
targs = TrainingArguments(
    output_dir=OUT_DIR,

    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    warmup_ratio=0.05,
    max_steps=MAX_STEPS,

    fp16=not bf16_ok,
    bf16=bf16_ok,

    optim="paged_adamw_8bit",

    logging_steps=10,
    eval_strategy="steps",   # you said you changed to eval_strategy; that's fine
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=4,

    # CRITICAL PARTS
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    gradient_checkpointing=True,  # keep enabled
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Trainer(
    model=model,
    args=targs,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=4)
    ],
    # tokenizer=tok,  # optional; warning is fine. keep or remove.
)

trainer.train()

#  [251/300 2:26:30 < 28:49, 0.03 it/s, Epoch 6.10/8]
# Step	Training Loss	Validation Loss
# 25	0.042300	0.002675
# 50	0.000600	0.001167
# 75	0.000300	0.000527
# 100	0.000100	0.000579
# 125	0.000100	0.000481
# 150	0.000000	0.000477
# 175	0.000100	0.000513
# 200	0.000100	0.000509
# 225	0.000000	0.000521
# [ 59/112 00:34 < 00:31, 1.66 it/s]

Step,Training Loss,Validation Loss
25,0.042300,0.002675
50,0.000600,0.001167
75,0.000300,0.000527
100,0.000100,0.000579
125,0.000100,0.000481
150,0.000000,0.000477
175,0.000100,0.000513
200,0.000100,0.000509
225,0.000000,0.000521


### Load saved checkpoints and prepare to reload trained LoRA model

- Reconnect Google Drive to the Colab session (so the saved model checkpoints and training outputs can be accessed).
- Search output directory for saved checkpoints and display most recent ones (identify which checkpoint will be used for loading/evaluation).
- Specify checkpoint directory to be used (e.g., the last or best checkpoint produced during training).


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Mounted at /content/drive

Mounted at /content/drive


In [ ]:
import os, glob

OUT_DIR = "/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab"

ckpts = sorted(glob.glob(os.path.join(OUT_DIR, "checkpoint-*")), key=lambda p: int(p.split("-")[-1]))
print("Found checkpoints:", ckpts[-10:])
print("Last checkpoint:", ckpts[-1] if ckpts else None)

# Found checkpoints: ['/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-150', '/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-200', '/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-225', '/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-250']
# Last checkpoint: /content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-250

Found checkpoints: ['/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-150', '/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-200', '/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-225', '/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-250']
Last checkpoint: /content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-250


In [ ]:
CKPT_DIR = "/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-250"
OUT_DIR  = "/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab"

- Apply a temporary monkey-patch to disable the `check_model_inputs` decorator and reload the Qwen2 model module.  

This workaround avoids compatibility issues when reloading fine-tuned model in current Transformers version.

In [ ]:
import transformers
import importlib

# 1) Monkey-patch: make check_model_inputs a no-op decorator
import transformers.utils.generic as generic

def _no_check_model_inputs(*args, **kwargs):
    def _decorator(fn):
        return fn
    return _decorator

generic.check_model_inputs = _no_check_model_inputs

# 2) Reload the Qwen2 modeling module so it re-defines forward() using the patched decorator
import transformers.models.qwen2.modeling_qwen2 as modeling_qwen2
importlib.reload(modeling_qwen2)

print("✅ Patched check_model_inputs and reloaded modeling_qwen2")


✅ Patched check_model_inputs and reloaded modeling_qwen2


### Reload the fine-tuned model (base + LoRA) and merge for inference

- Load the original base model and tokenizer,
- then load the LoRA adapter weights from a selected checkpoint.  
- finally, merge the LoRA adapters into the base model (`merge_and_unload`) to produce a single inference-ready model and switch it to evaluation mode.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL = "Qwen/Qwen2.5-7B-Instruct"
CKPT_DIR = "/content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-150"

tok = AutoTokenizer.from_pretrained(MODEL, use_fast=True, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa",
)

# Load adapter
peft_model = PeftModel.from_pretrained(base, CKPT_DIR)

# Merge LoRA into base weights for inference stability
model = peft_model.merge_and_unload()
model.eval()

print("✅ Loaded + merged adapter from:", CKPT_DIR)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 55804009-23b5-4e21-864c-7c8b8523775b)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-7B-Instruct/resolve/main/vocab.json
Retrying in 1s [Retry 1/5].


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Loaded + merged adapter from: /content/drive/MyDrive/notebooks/projectz_at_merce/twingpt/qwen2p5_sft_lora_colab/checkpoint-150


### Run inference

- Provide a natural-language AMI deployment request,
- Format it using the model’s chat template, and
- Generate the corresponding XML configuration.  

This step evaluates whether the fine-tuned model can correctly translate a scenario description into a valid XML output.

In [ ]:
prompt_msgs = [
    {"role": "system", "content": "Translate an AMI scenario request into a valid XML following TopologySchema.xsd. Output XML only."},
    {"role": "user", "content": "Simulate an AMI neighborhood with 3 smart meters sending 64-byte reports every 20 seconds over Wi-Fi to a collector. Collector forwards to headend over 1 Mbps backhaul with 15 ms delay. Run from time 2 to 120. Reliability 99%, latency 10 seconds."},
]

prompt_text = tok.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=1200,
        do_sample=False,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
    )

print(tok.decode(out[0], skip_special_tokens=True))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


system
Translate an AMI scenario request into a valid XML following TopologySchema.xsd. Output XML only.
user
Simulate an AMI neighborhood with 3 smart meters sending 64-byte reports every 20 seconds over Wi-Fi to a collector. Collector forwards to headend over 1 Mbps backhaul with 15 ms delay. Run from time 2 to 120. Reliability 99%, latency 10 seconds.
assistant
<?xml version="1.0" encoding="UTF-8"?><Gen><Nodes><node><type>Station</type><name>station_0</name><enableFlowmonitor>true</enableFlowmonitor></node><node><type>Station</type><name>station_1</name><enableFlowmonitor>true</enableFlowmonitor></node><node><type>Station</type><name>station_2</name><enableFlowmonitor>true</enableFlowmonitor></node><node><type>AccessPoint</type><name>ap_0</name><enableFlowmonitor>true</enableFlowmonitor></node><node><type>Pc</type><name>term_0</name><enableFlowmonitor>true</enableFlowmonitor></node></Nodes><NetworkHardwares><networkHardware><type>Ap</type><name>hwap_0</name><dataRate>100000000</data

### Example generated XML (model output)

Below is a sample output produced by the fine-tuned model for the given AMI request.  

Input describtion

```
Translate an AMI scenario request into a valid XML following TopologySchema.xsd. Output XML only.
```

user


> _Simulate an AMI neighborhood with 3 smart meters sending 64-byte reports every 20 seconds over Wi-Fi to a collector. Collector forwards to headend over 1 Mbps backhaul with 15 ms delay. Run from time 2 to 120. Reliability 99%, latency 10 seconds_


```xml
<?xml version="1.0" encoding="UTF-8"?><Gen><Nodes><node><type>Station</type><name>station_0</name><enableFlowmonitor>true</enableFlowmonitor></node><node><type>Station</type><name>station_1</name><enableFlowmonitor>true</enableFlowmonitor></node><node><type>Station</type><name>station_2</name><enableFlowmonitor>true</enableFlowmonitor></node><node><type>AccessPoint</type><name>ap_0</name><enableFlowmonitor>true</enableFlowmonitor></node><node><type>Pc</type><name>term_0</name><enableFlowmonitor>true</enableFlowmonitor></node></Nodes><NetworkHardwares><networkHardware><type>Ap</type><name>hwap_0</name><dataRate>100000000</dataRate><linkDelay>10000</linkDelay><enableTrace>false</enableTrace><connectedNodes><name>ap_0</name><name>station_0</name><name>station_1</name><name>station_2</name></connectedNodes></networkHardware><networkHardware><type>Hub</type><name>hub_0</name><dataRate>100000000</dataRate><linkDelay>6.56</linkDelay><enableTrace>false</enableTrace><connectedNodes><name>term_0</name></connectedNodes></networkHardware><networkHardware><type>PointToPoint</type><name>p2p_0</name><dataRate>1000000</dataRate><linkDelay>15</linkDelay><enableTrace>false</enableTrace><connectedNodes><name>ap_0</name><name>term_0</name></connectedNodes></networkHardware></NetworkHardwares><Applications><application><type>UdpEcho</type><name>udpEcho_0</name><sender>station_0</sender><receiver>term_0</receiver><startTime>2</startTime><endTime>120</endTime><special><port>111</port><packetSize>64</packetSize><maxPacketCount>100</maxPacketCount><packetIntervalTime>20</packetIntervalTime></special></application><application><type>UdpEcho</type><name>udpEcho_1</name><sender>station_1</sender><receiver>term_0</receiver><startTime>2</startTime><endTime>120</endTime><special><port>112</port><packetSize>64</packetSize><maxPacketCount>100</maxPacketCount><packetIntervalTime>20</packetIntervalTime></special></application><application><type>UdpEcho</type><name>udpEcho_2</name><sender>station_2</sender><receiver>term_0</receiver><startTime>2</startTime><endTime>120</endTime><special><port>113</port><packetSize>64</packetSize><maxPacketCount>100</maxPacketCount><packetIntervalTime>20</packetIntervalTime></special></application></Applications><Flows><flow><type>WifiFlow</type><name>wifi_flow_0</name><source>station_0</source><destination>term_0</destination><expectedDelaySeconds>10</expectedDelaySeconds><expectedReliabilityPercent>0.99</expectedReliabilityPercent></flow><flow><type>WifiFlow</type><name>wifi_flow_1</name><source>station_1</source><destination>term_0</destination><expectedDelaySeconds>10</expectedDelaySeconds><expectedReliabilityPercent>0.99</expectedReliabilityPercent></flow><flow><type>WifiFlow</type><name>wifi_flow_2</name><source>station_2</source><destination>term_0</destination><expectedDelaySeconds>10</expectedDelaySeconds><expectedReliabilityPercent>0.99</expectedReliabilityPercent></flow></Flows></Gen>
```